# Participant eligibility by reported age

Use this notebook to inspect or regenerate the age decisions used throughout the analysis. Run all cells from a fresh kernel. All inputs are included, so no other notebook needs to run first.

**Inputs:** `data/raw/{country}_metadata.csv`, the three `*_sample_reference_ids.csv` membership lists, and `chile_all_source_ids.csv`. The membership lists contain IDs only, not fitted coordinates.

**Outputs:** `data/processed/participant_age_decisions.csv` and age summaries in `outputs/tables/`. Notebooks 02 and 03 read these decisions before training.

Chile excludes reported ages below 19 or above 120. France and Brazil record age bands, so the whole 10–19 band is excluded. Codes 98 (Other) and 99 (Skip), missing answers and absent age questionnaires remain eligible as unspecified ages. Conflicting or invalid ages are excluded. Age is never a DYNAMAP predictor.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/raw").is_dir(), "Open from the project root or notebooks directory."
RAW = ROOT / "data/raw"
PROCESSED = ROOT / "data/processed"
TABLES = ROOT / "outputs/tables"
PROCESSED.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# Only Chile reports exact ages. France and Brazil use the band codes below.
CHILE_MINIMUM_AGE = 19
CHILE_MAXIMUM_AGE = 120
UNSPECIFIED_AGE_CODES = [98, 99]


In [2]:
decisions = []
main_fit_ids = {}
for country in ("Chile", "France", "Brazil"):
    metadata = pd.read_csv(RAW / f"{country.lower()}_metadata.csv")
    key = "uuid" if country == "Chile" else "user_id"
    original = pd.read_csv(RAW / f"{country.lower()}_sample_reference_ids.csv")
    original_key = next(
        key for key in ("uuid", "user_id", "participant_id") if key in original.columns
    )
    assert original[original_key].is_unique
    main_fit_ids[country] = set(original[original_key])
    metadata = metadata.rename(columns={key: "uuid"})[["uuid", "age"]].drop_duplicates()
    assert metadata.uuid.notna().all()
    ages_by_id = metadata.groupby("uuid", sort=False).age
    # The union keeps participants whose demographic questionnaire is absent.
    universe = set(metadata.uuid) | main_fit_ids[country]
    if country == "Chile":
        universe |= set(pd.read_csv(RAW / "chile_all_source_ids.csv").uuid)
    universe = sorted(universe)
    for uuid in universe:
        raw_ages = (
            ages_by_id.get_group(uuid) if uuid in ages_by_id.groups else pd.Series(dtype=float)
        )
        numeric = pd.to_numeric(raw_ages, errors="coerce")
        malformed = (raw_ages.notna() & numeric.isna()).any()
        if country != "Chile":
            numeric = numeric.mask(numeric.isin(UNSPECIFIED_AGE_CODES))
        # Distinct valid reports determine conflicts, while unspecified codes do not.
        reported = numeric.dropna().unique()
        age = float(reported[0]) if len(reported) == 1 else np.nan
        if malformed:
            status = "invalid_age"
        elif len(reported) == 0:
            status = "unknown_age_retained"
        elif len(reported) > 1:
            status = "conflicting_age"
        elif country == "Chile":
            if not np.isfinite(age) or not (1 <= age <= CHILE_MAXIMUM_AGE):
                status = "invalid_age"
            elif age < CHILE_MINIMUM_AGE:
                status = "below_age_cutoff"
            else:
                status = "eligible_reported_age"
        # France/Brazil: code 1 is 10–19, codes 2–7 are the older age bands.
        elif age == 1:
            status = "below_age_cutoff"
        elif age in range(2, 8):
            status = "eligible_reported_age"
        else:
            status = "invalid_age"
        include = status in ("unknown_age_retained", "eligible_reported_age")
        decisions.append(
            {
                "country": country,
                "uuid": uuid,
                "age": age,
                "include": include,
                "status": status,
                "in_original_main_fit": uuid in main_fit_ids[country],
            }
        )

decisions = pd.DataFrame(decisions)
assert not decisions.duplicated(["country", "uuid"]).any()
assert decisions.loc[decisions.status.eq("unknown_age_retained"), "include"].all()
assert not decisions.loc[
    decisions.status.isin(["invalid_age", "conflicting_age", "below_age_cutoff"]), "include"
].any()


## Summarize the decisions

The summaries below use the original main-sample membership lists. The full decision table also covers Chile's other source IDs. Training notebooks remove all choices from excluded IDs and retain participants whose age is unknown.


In [3]:
sample_counts = (
    decisions.loc[decisions.in_original_main_fit]
    .groupby(["country", "status", "include"], observed=True)
    .size()
    .rename("participants")
    .reset_index()
)
display(sample_counts)
sample_counts.to_csv(TABLES / "age_exclusion_counts.csv", index=False)
sample_counts.to_latex(TABLES / "age_exclusion_counts.tex", index=False)

display(decisions)
decisions.to_csv(PROCESSED / "participant_age_decisions.csv", index=False)
retained_counts = (
    decisions.loc[decisions.in_original_main_fit]
    .groupby("country", observed=True)
    .agg(original_participants=("uuid", "size"), retained_participants=("include", "sum"))
    .reset_index()
)
retained_counts["excluded_participants"] = (
    retained_counts.original_participants - retained_counts.retained_participants
)
retained_counts["excluded_percent"] = (
    100 * retained_counts.excluded_participants / retained_counts.original_participants
)
display(retained_counts)
retained_counts.to_csv(TABLES / "age_exclusion_summary.csv", index=False)
retained_counts.to_latex(TABLES / "age_exclusion_summary.tex", index=False, float_format="%.3f")


,country,status,include,participants
0,Brazil,below_age_cutoff,False,25
1,Brazil,eligible_reported_age,True,648
2,Brazil,unknown_age_retained,True,117
3,Chile,below_age_cutoff,False,395
4,Chile,conflicting_age,False,88
5,Chile,eligible_reported_age,True,37636
6,Chile,invalid_age,False,62
7,Chile,unknown_age_retained,True,9938
8,France,below_age_cutoff,False,45
9,France,eligible_reported_age,True,1117


Matplotlib is building the font cache; this may take a moment.


,country,uuid,age,include,status,in_original_main_fit
0,Chile,00004bef-1591-46fb-be1b-2aaa897584ce,NaN,True,unknown_age_retained,False
1,Chile,000057c7-295c-4a61-8f62-76aff62bf079,27.0,True,eligible_reported_age,True
2,Chile,000059b8-10b9-4f74-9c01-80bd2c48f509,37.0,True,eligible_reported_age,True
3,Chile,0000d93e-3301-4ec7-ba33-e9ecc8bdd77d,18.0,False,below_age_cutoff,False
4,Chile,000101ba-3ebf-4231-b6e9-901e41ac6ba9,NaN,True,unknown_age_retained,False
...,...,...,...,...,...,...
131227,Brazil,fe7adca8-9f18-4048-bd9a-a0d1c7636378,NaN,True,unknown_age_retained,True
131228,Brazil,fe8ee69a-b37b-4e1b-be58-9391d29f0727,5.0,True,eligible_reported_age,True
131229,Brazil,fecc1142-5009-42bc-b2c8-389c77e7b1fd,3.0,True,eligible_reported_age,True
131230,Brazil,fed0ccb0-fdf8-423a-abd1-cef80cc4d6c5,3.0,True,eligible_reported_age,True


,country,original_participants,retained_participants,excluded_participants,excluded_percent
0,Brazil,790,765,25,3.164557
1,Chile,48119,47574,545,1.132609
2,France,1488,1443,45,3.024194
